In [94]:
import polars as pl
import pandas as pd
import numpy as mp
import altair as alt
import pyarrow
from pyarrow import dictionary

from func.UsefulFunctions import *

In [95]:
groupDF = pl.read_parquet("Data/pipelinedGroupsWikidata.parquet")
groupDF.head()

group,groupLabel,genre,groupType,inception,countryOfOrigin,recordLabel,genreLabel,groupTypeLabel,countryOfOriginLabel,recordLabelLabel,influencedby,influencedbyLabel,musicBrainzID,article,statementCount,__index_level_0__
str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,f64,i64
"""Q72092""","""Rage Against the Machine""","""Q877693""","""Q215380""",2001.0,"""Q30""","""Q216364""","""rap rock""","""musical group""","""United States""","""Epic Records""",null,null,"""3798b104-01cb-484c-a3b0-56adc6…","""https://en.wikipedia.org/wiki/…",125.0,0
"""Q72092""","""Rage Against the Machine""","""Q877693""","""Q215380""",2001.0,"""Q30""","""Q1759336""","""rap rock""","""musical group""","""United States""","""Revelation Records""",null,null,"""3798b104-01cb-484c-a3b0-56adc6…","""https://en.wikipedia.org/wiki/…",125.0,1
"""Q72349""","""Krewella""","""Q20474""","""Q14073567""",2010.0,"""Q30""","""Q183387""","""dubstep""","""sibling duo""","""United States""","""Columbia Records""",null,null,"""d87475fa-76a1-4e12-962c-fc9941…","""https://en.wikipedia.org/wiki/…",73.0,2
"""Q72349""","""Krewella""","""Q20474""","""Q14073567""",2010.0,"""Q30""","""Q10398667""","""dubstep""","""sibling duo""","""United States""","""Monstercat""",null,null,"""d87475fa-76a1-4e12-962c-fc9941…","""https://en.wikipedia.org/wiki/…",73.0,3
"""Q72349""","""Krewella""","""Q20502""","""Q14073567""",2010.0,"""Q30""","""Q183387""","""house music""","""sibling duo""","""United States""","""Columbia Records""",null,null,"""d87475fa-76a1-4e12-962c-fc9941…","""https://en.wikipedia.org/wiki/…",73.0,4


In [96]:
groupDF.schema

Schema([('group', String),
        ('groupLabel', String),
        ('genre', String),
        ('groupType', String),
        ('inception', Float64),
        ('countryOfOrigin', String),
        ('recordLabel', String),
        ('genreLabel', String),
        ('groupTypeLabel', String),
        ('countryOfOriginLabel', String),
        ('recordLabelLabel', String),
        ('influencedby', String),
        ('influencedbyLabel', String),
        ('musicBrainzID', String),
        ('article', String),
        ('statementCount', Float64),
        ('__index_level_0__', Int64)])

In [97]:
#Look at the size of the dataset
size = groupDF.select(pl.col("group")).unique()
print(f"There are {len(size)} groups in this dataset")

There are 125699 groups in this dataset


In [98]:
#What are the unique group types
groupDF.select(
    pl.col(["groupType", "groupTypeLabel"])
).unique().write_csv("variousGroupTypes.csv")


In [99]:
#Drop everyone who is fictitious as no characters are allowed
#Get all the people who have have fictitious in there
characterEntities= groupDF.filter(
    pl.col("groupTypeLabel").str.contains("(?i)fictional")
).select(pl.col("group")).unique()

#drop everyone who is not fictional and deduplicate
groupDF = groupDF.filter(
    ~pl.col("group").is_in(characterEntities["group"].implode())
).unique()


print(f"There are {len(groupDF["group"].unique())} groups in this dataset")




There are 125650 groups in this dataset


In [100]:
def nullColumnLabelWithissues(df, column):
    columnLabel = column + "Label"

    noLabel = df.filter(
        pl.col(column).is_in(pl.col(columnLabel).implode())
    ).select(pl.col(columnLabel)).unique()
    noLabel_dict= {k: None for k in noLabel[columnLabel].to_list()}
    df = df.with_columns(pl.col(columnLabel).replace(noLabel_dict))

    nolabelDF = df.filter(
        pl.col(columnLabel).is_null(),
    ).select(pl.col(["group","groupLabel"])).unique()
    display(nolabelDF)
    print(f"There are {len(nolabelDF)} groups with no default labels/labels in english in Column: {columnLabel}.")
    if column == "genre":
        nolabelDF.write_csv(f"./missingValuesStore/groupEntitiesWithoutLabels.csv")
    else:
        nolabelDF.write_csv(f"./missingValuesStore/groupEntitiesWithoutLabelled{column}OrNo{column}.csv")

    return df


In [101]:
#Null the group name labels for the groups who do not have a group label
groupDF = nullColumnLabelWithissues(groupDF, "group")

group,groupLabel
str,str
"""Q102227234""",null
"""Q20549131""",null
"""Q135926727""",null
"""Q104600126""",null
"""Q123361719""",null
…,…
"""Q124423013""",null
"""Q97177539""",null
"""Q11868466""",null


There are 15927 groups with no default labels/labels in english in Column: groupLabel.


In [102]:
#Check the genres and the genreLabels - Cleaned previously in the pipline stage, to remove error ones
#No genre added had no name
groupDF.filter(
    pl.col("genre").is_in(pl.col("genreLabel").implode())
)
#create a list of artists with no labelled genres or no genres
entitiesWithNullGenres = groupDF.filter(
    pl.col("genreLabel").is_null()
).select(pl.col(["group","groupLabel"])).unique()
display(entitiesWithNullGenres)
entitiesWithNullGenres.write_csv("./missingValuesStore/groupEntitiesWithoutLabelledGenresOrNoGenres.csv")


group,groupLabel
str,str
"""Q65059472""","""Rochell & the Candles"""
"""Q129518311""","""Pissoff"""
"""Q7261945""","""Pussyfoot"""
"""Q18915467""","""Malina Brothers"""
"""Q12019554""","""Heroldovo kvarteto"""
…,…
"""Q137770196""",null
"""Q6005352""","""Immortal Dominion"""
"""Q97224061""",null


In [103]:
groupDF.filter(
    pl.col("genre").is_in(pl.col("genreLabel").implode())
)

group,groupLabel,genre,groupType,inception,countryOfOrigin,recordLabel,genreLabel,groupTypeLabel,countryOfOriginLabel,recordLabelLabel,influencedby,influencedbyLabel,musicBrainzID,article,statementCount,__index_level_0__
str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,f64,i64


In [104]:
#Check the group Types. In pipeline stage already removed groupTypes that had no Q number from pipeline
groupDF = nullColumnLabelWithissues(groupDF, "groupType")

group,groupLabel
str,str
"""Q110687782""",null
"""Q15825121""",null
"""Q12085520""",null
"""Q1802266""",null
"""Q3693814""",null
…,…
"""Q1593588""","""Heeresmusikkorps 12"""
"""Q812421""","""Bayerisches Landesjugendorches…"
"""Q139586671""",null


There are 64 groups with no default labels/labels in english in Column: groupTypeLabel.


In [105]:
#Check the countryOfOrign. In pipeline stage already removed countryOfOrigin that had no Q number from earlier
groupDF = nullColumnLabelWithissues(groupDF, "countryOfOrigin")

group,groupLabel
str,str
"""Q50285640""",null
"""Q130634893""",null
"""Q7713186""","""The Alligators"""
"""Q3427463""","""Reptiles at Dawn"""
"""Q6582371""","""Szeged Chamber Orchestra"""
…,…
"""Q105474341""","""The Rhythmettes"""
"""Q104855128""","""Crisis n.T.i."""
"""Q11858791""",null


There are 34900 groups with no default labels/labels in english in Column: countryOfOriginLabel.


In [106]:
#Check the recordLabel and influencedby. As with the other fields the non-Q-numbers were removed
groupDF = nullColumnLabelWithissues(groupDF, "recordLabel")
groupDF = nullColumnLabelWithissues(groupDF, "influencedby")

group,groupLabel
str,str
"""Q11951791""","""Three Dead Trolls in a Baggie"""
"""Q16138945""","""Proud"""
"""Q25490225""","""Orpheus Blade"""
"""Q41546276""","""Leichenzug"""
"""Q109441321""","""Sparkling Saint"""
…,…
"""Q12010037""",null
"""Q115814479""","""Levee Town"""
"""Q12255276""","""Birracel"""


There are 101923 groups with no default labels/labels in english in Column: recordLabelLabel.


group,groupLabel
str,str
"""Q4861176""","""Barleyjuice"""
"""Q124004724""","""Codigo FN"""
"""Q51029693""","""Onirik"""
"""Q107352327""","""Apichat Pakwan"""
"""Q6134425""","""Storm"""
…,…
"""Q3552203""","""Urban Waste"""
"""Q7595661""","""St Winifred's School Choir"""
"""Q38266682""","""Sadistik Forest"""


There are 125650 groups with no default labels/labels in english in Column: influencedbyLabel.


In [107]:
#Look at the musicBrainz IDs to confirm that there isn't an issue observed, and it was fixed in the pipeline
groupDF.filter(
    (pl.col("musicBrainzID").str.contains("http")) | (pl.col("musicBrainzID").str.contains("user")) | (pl.col("musicBrainzID").str.contains("artist"))
)

group,groupLabel,genre,groupType,inception,countryOfOrigin,recordLabel,genreLabel,groupTypeLabel,countryOfOriginLabel,recordLabelLabel,influencedby,influencedbyLabel,musicBrainzID,article,statementCount,__index_level_0__
str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,f64,i64


In [108]:
#Looked at the article column to confirm that the filters in the pipeline column worked correctly and only returned proper links
groupDF.filter(
    ~pl.col("article").str.contains("(?i)en.wikipedia")
)

group,groupLabel,genre,groupType,inception,countryOfOrigin,recordLabel,genreLabel,groupTypeLabel,countryOfOriginLabel,recordLabelLabel,influencedby,influencedbyLabel,musicBrainzID,article,statementCount,__index_level_0__
str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,f64,i64


In [109]:
#Change the inception column to years
groupsWithNoInception = groupDF.filter(
    pl.col("inception").is_null()
).select(['group','groupLabel']).unique()
print(f"There are {len(groupsWithNoInception)} groups with no inception year.")
groupsWithNoInception.write_csv("./missingValuesStore/groupEntitiesWithNoInceptionYear.csv")

There are 76962 groups with no inception year.


In [110]:
groupDF.write_parquet("./Data/Network_groupWikidata.parquet")